# Clustering de estudios usando CRISP-DM

Esta libreta muestra el flujo completo de la propuesta de
**clustering**:

- cargar el dataset
- seleccionar variables numéricas
- escalar los datos
- evaluar distintos valores de `k`
- entrenar `KMeans`
- exportar el modelo a `.pkl`


## 1. Comprensión del negocio

El objetivo es encontrar grupos de estudios con comportamientos
operativos similares sin usar una etiqueta previa.


## 2. Comprensión de los datos

Primero se carga el dataset y se revisa su estructura general.


In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

BACKEND = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "package.json").exists() and (path / "src").exists()
)


Librerías cargadas correctamente.


In [2]:
df = pd.read_csv(BACKEND / "05_Datasets" / "clustering_estudios.csv")
print(df.head().to_string(index=False))


study_id    code                           name  price  delivery_hours  parameter_count  request_count  synthetic_request_count sample_type    analysis_method requires_special_processing  is_synthetic
       1 GLU-001                        GLUCOSA    110            1.00              NaN              5                        1         NaN                NaN                         NaN         False
       2    BH01             BIOMETRIA HEMATICA    180            0.50              NaN              3                        1         NaN       AUTOMATIZADO                         NaN         False
       3    QS06  QUIMICA SANGUINEA 6 ELEMENTOS    220            0.75              NaN              0                        0         NaN ESPECTROFOTOMETRIA                         NaN         False
       4    QS12 QUIMICA SANGUINEA 12 ELEMENTOS    350            1.00              NaN              4                        2         NaN ESPECTROFOTOMETRIA                         NaN         F

In [3]:
print(df.shape)


(2000, 12)


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   study_id                     2000 non-null   int64  
 1   code                         2000 non-null   str    
 2   name                         2000 non-null   str    
 3   price                        2000 non-null   int64  
 4   delivery_hours               2000 non-null   float64
 5   parameter_count              1953 non-null   float64
 6   request_count                2000 non-null   int64  
 7   synthetic_request_count      2000 non-null   int64  
 8   sample_type                  1958 non-null   str    
 9   analysis_method              1999 non-null   str    
 10  requires_special_processing  1958 non-null   object 
 11  is_synthetic                 2000 non-null   bool   
dtypes: bool(1), float64(2), int64(4), object(1), str(4)
memory usage: 174.0+ KB


In [5]:
print(df[["price", "delivery_hours", "parameter_count", "request_count"]].describe().round(2).to_string())


         price  delivery_hours  parameter_count  request_count
count  2000.00         2000.00          1953.00        2000.00
mean    600.32            3.59            15.51           1.19
std     230.39            3.55             8.70           1.12
min      80.00            0.25             1.00           0.00
25%     420.00            1.75             8.00           0.00
50%     600.00            2.50            15.00           1.00
75%     770.00            3.50            23.00           2.00
max    1260.00           48.00            32.00           5.00


In [6]:
print(
    df[
        [
            "price",
            "delivery_hours",
            "parameter_count",
            "request_count",
            "sample_type",
            "analysis_method",
        ]
    ].isnull().sum().to_string()
)


price               0
delivery_hours      0
parameter_count    47
request_count       0
sample_type        42
analysis_method     1


### Descripción de las variables

Para esta propuesta se usan variables numéricas sencillas:

- `price`
- `delivery_hours`
- `parameter_count`
- `request_count`


## 3. Preparación de los datos

Se seleccionan las variables del clustering, se imputan nulos con la
mediana y después se escalan los datos.


In [7]:
features = ["price", "delivery_hours", "parameter_count", "request_count"]

X = df[features].copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables usadas:", features)
print("Filas útiles:", len(X))


Variables usadas: ['price', 'delivery_hours', 'parameter_count', 'request_count']
Filas útiles: 2000


## 4. Modelado: selección de la cantidad de clusters

Se prueban varios valores de `k` y se revisan tres evidencias:

- inercia
- silhouette
- Davies-Bouldin


In [8]:
resultados = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    resultados.append(
        {
            "k": k,
            "inertia": round(float(model.inertia_), 4),
            "silhouette": round(float(silhouette_score(X_scaled, labels)), 4),
            "davies_bouldin": round(float(davies_bouldin_score(X_scaled, labels)), 4),
        }
    )

resultados = pd.DataFrame(resultados)
print(resultados.to_string(index=False))


k   inertia  silhouette  davies_bouldin
2 5170.9578      0.3387          1.1965
3 4135.4452      0.3578          1.1498
4 3346.8123      0.3430          1.0588
5 2769.8232      0.3238          1.0944
6 2436.8840      0.3073          1.0203


In [9]:
best_k = int(
    resultados.sort_values(
        ["silhouette", "davies_bouldin"],
        ascending=[False, True],
    ).iloc[0]["k"]
)
print("k seleccionado:", best_k)


k seleccionado: 3


## 5. Entrenamiento final y asignación de clusters

Se entrena el modelo final con el valor de `k` seleccionado y se
calcula el cluster de cada estudio.


In [10]:
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = final_model.fit_predict(X_scaled)
distances = final_model.transform(X_scaled).min(axis=1)

thresholds = {}
for cluster in range(best_k):
    cluster_distances = np.sort(distances[labels == cluster])
    q1 = np.quantile(cluster_distances, 0.25)
    q3 = np.quantile(cluster_distances, 0.75)
    iqr = q3 - q1
    thresholds[cluster] = q3 + 1.5 * iqr if iqr > 0 else q3

asignaciones = df[["study_id", "code", "name"]].copy()
asignaciones["cluster"] = labels + 1
asignaciones["distance_to_centroid"] = np.round(distances, 4)
asignaciones["is_outlier"] = [
    bool(distances[i] > thresholds[labels[i]])
    for i in range(len(labels))
]

print(asignaciones.head(8).to_string(index=False))


study_id    code                           name  cluster  distance_to_centroid  is_outlier
       1 GLU-001                        GLUCOSA        1                3.7706        True
       2    BH01             BIOMETRIA HEMATICA        1                2.1025       False
       3    QS06  QUIMICA SANGUINEA 6 ELEMENTOS        1                1.5179       False
       4    QS12 QUIMICA SANGUINEA 12 ELEMENTOS        1                2.6898        True
       5   GLU01                        GLUCOSA        1                3.8353        True
       6   HBA1C  Hemoglobina glucosilada HbA1c        1                1.3679       False
       7   COL01               COLESTEROL TOTAL        1                1.6110       False
       8   TRI01                  TRIGLICERIDOS        1                3.0280        True


## 6. Exportación del modelo

El modelo entrenado y el escalador se guardan en `.pkl`, mientras
que las asignaciones del conjunto completo se exportan a CSV.


In [11]:
model_path = BACKEND / "07_Modelos" / "clustering_estudios_model.pkl"
assignments_path = BACKEND / "05_Datasets" / "clustering_estudios_asignaciones.csv"

bundle = {
    "model": final_model,
    "scaler": scaler,
    "features": features,
    "outlier_thresholds": thresholds,
}

with model_path.open("wb") as file:
    pickle.dump(bundle, file)

asignaciones.to_csv(assignments_path, index=False)

print("Modelo exportado en:", model_path)
print("Asignaciones guardadas en:", assignments_path)


Modelo exportado en: C:\Users\Josafat Tapia\Desktop\Econolab_Escuela\Backend_Econolab_Escuela\07_Modelos\clustering_estudios_model.pkl
Asignaciones guardadas en: C:\Users\Josafat Tapia\Desktop\Econolab_Escuela\Backend_Econolab_Escuela\05_Datasets\clustering_estudios_asignaciones.csv


## 7. Ejemplo de asignación

Como salida final, cada estudio queda asociado a un cluster.


In [12]:
print(asignaciones.iloc[0][["study_id", "cluster", "distance_to_centroid", "is_outlier"]].to_dict())


{
  "study_id": 1,
  "cluster": 1,
  "distance_to_centroid": 3.7706,
  "is_outlier": true
}
